In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class HeteroatomOxidation(MorphingOperator):
    def __init__(self):
        super(HeteroatomOxidation, self).__init__()
        self._name = "Heteroatom Oxidation (Phase I)"
        self._matches = []
        self.N_PATTERN = Chem.MolFromSmarts("[N;X3;H0;!$(N-C=O);!a](-[#6])-[#6]")
        self.S_THIOETHER = Chem.MolFromSmarts("[S;X2;H0;!a]")
        self.S_SULFOXIDE = Chem.MolFromSmarts("[S;X3;D3;H0](=O)")

    def setOriginal(self, mol):
        super(HeteroatomOxidation, self).setOriginal(mol)
        self._matches = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        if self.N_PATTERN is not None:
            for match in rdkit_mol.GetSubstructMatches(self.N_PATTERN):
                self._matches.append((match[0], "N"))

        if self.S_THIOETHER is not None:
            for match in rdkit_mol.GetSubstructMatches(self.S_THIOETHER):
                self._matches.append((match[0], "S_thio"))

        if self.S_SULFOXIDE is not None:
            for match in rdkit_mol.GetSubstructMatches(self.S_SULFOXIDE):
                self._matches.append((match[0], "S_sulf"))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        target_idx, atom_type = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            oxygen_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            target_atom = rw_mol.GetAtomWithIdx(target_idx)
            ox_atom = rw_mol.GetAtomWithIdx(oxygen_idx)

            if atom_type in ["S_thio", "S_sulf"]:

                rw_mol.AddBond(target_idx, oxygen_idx, Chem.BondType.DOUBLE)
                
                target_atom.SetNoImplicit(False)
                target_atom.SetNumExplicitHs(0)
                
            elif atom_type == "N":
                
                rw_mol.AddBond(target_idx, oxygen_idx, Chem.BondType.SINGLE)
                target_atom.SetFormalCharge(1)
                ox_atom.SetFormalCharge(-1)
                ox_atom.SetNoImplicit(True)
                ox_atom.SetNumExplicitHs(0)
                target_atom.SetNoImplicit(False)
                target_atom.SetNumExplicitHs(0)

            new_mol = rw_mol.GetMol()
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name

hetero_op = HeteroatomOxidation()

test_hetero_molecules = {
    "1. Τριμεθυλαμίνη (Τριτοταγής Αμίνη -> N-Oxide)": "CN(C)C",
    "2. Διμεθυλοθειοαιθέρας (Θειοαιθέρας -> Σουλφοξείδιο)": "CSC",
    "3. Διμεθυλοσουλφόνη (Σουλφόνη -> Πρέπει να αγνοηθεί - STOP VALVE)": "CS(=O)(=O)C",
    "4. DMF (Αμίδιο με Άζωτο -> Πρέπει να αγνοηθεί)": "CN(C)C=O",
    "5. DMSO (Σουλφοξείδιο -> Σουλφόνη)" : "CS(C)=O",
    "6. Παγίδα Ετεροατόμου (NF3 - Όχι αμίνη -> Πρέπει να αγνοηθεί)": "FN(F)F"
}


print("=== STARTING HETEROATOM OXIDATION TESTING ===")
for name, smiles in test_hetero_molecules.items():
    mol = MolpherMol(smiles)
    hetero_op.setOriginal(mol)
    product = hetero_op.morph()
    
    print(f"\n{name}")
    print(f"  SOURCE: {mol.getSMILES()}")
    print(f"  TARGET: {product.getSMILES() if product and product.getSMILES() != mol.getSMILES() else 'No change (Safe)'}")
print("\n=============================================")

=== STARTING HETEROATOM OXIDATION TESTING ===

1. Τριμεθυλαμίνη (Τριτοταγής Αμίνη -> N-Oxide)
  SOURCE: CN(C)C
  TARGET: C[N+](C)(C)[O-]

2. Διμεθυλοθειοαιθέρας (Θειοαιθέρας -> Σουλφοξείδιο)
  SOURCE: CSC
  TARGET: CS(C)=O

3. Διμεθυλοσουλφόνη (Σουλφόνη -> Πρέπει να αγνοηθεί - STOP VALVE)
  SOURCE: CS(C)(=O)=O
  TARGET: No change (Safe)

4. DMF (Αμίδιο με Άζωτο -> Πρέπει να αγνοηθεί)
  SOURCE: CN(C)C=O
  TARGET: No change (Safe)

5. DMSO (Σουλφοξείδιο -> Σουλφόνη)
  SOURCE: CS(C)=O
  TARGET: CS(C)(=O)=O

6. Παγίδα Ετεροατόμου (NF3 - Όχι αμίνη -> Πρέπει να αγνοηθεί)
  SOURCE: FN(F)F
  TARGET: No change (Safe)

